# KG1 V198 micro-distillation Colab Pro run

Short continuation after V195. It trains from the best adapter found in Drive and never submits to Kaggle automatically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import importlib.util, os, pathlib, shutil, subprocess, sys, urllib.request, zipfile, hashlib
ROOT = pathlib.Path('/content/kg1_v198')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V198')
PACK = DRIVE_ROOT / 'kg1_v198_colab_pack.zip'
PACK_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/31d439bc4a9b33b7b3c772d3526149847103a9b1/runs/v198_micro_distill_colab_pack_20260503/kg1_v198_colab_pack.zip'
PACK_SHA256 = 'e61908c0f75018b0d265c3668600170f6fa99a1a4d559508f489cba9cd6b7c93'
OUT = DRIVE_ROOT / 'output_v198'
BASELINE_DIR = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V195/init_adapter/final')
V195_OUT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V195/output_v195')
BASE_ADAPTER_MODEL_SHA256 = '3d16ba908a5c8808624f1abd8fdc2b29f92723f5c874761161c894d7e5759f21'
BASE_ADAPTER_CONFIG_SHA256 = 'e5499f128fde60d32d0595d427e4fe84d8abe6dbde1d80886c970e8184e4b743'

def sha256_path(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def adapter_ready(path):
    cfg = path / 'adapter_config.json'
    model = path / 'adapter_model.safetensors'
    if not cfg.exists() or not model.exists():
        return False
    if cfg.stat().st_size < 100 or model.stat().st_size < 1024:
        return False
    try:
        import json
        json.loads(cfg.read_text(encoding='utf-8'))
    except Exception:
        return False
    return True

def ensure_baseline_adapter():
    BASELINE_DIR.mkdir(parents=True, exist_ok=True)
    if adapter_ready(BASELINE_DIR):
        cfg_ok = sha256_path(BASELINE_DIR / 'adapter_config.json') == BASE_ADAPTER_CONFIG_SHA256
        model_ok = sha256_path(BASELINE_DIR / 'adapter_model.safetensors') == BASE_ADAPTER_MODEL_SHA256
        if cfg_ok and model_ok:
            return BASELINE_DIR
        print('Existing baseline adapter has SHA mismatch; deleting and redownloading fallback.')
        for p in [BASELINE_DIR / 'adapter_config.json', BASELINE_DIR / 'adapter_model.safetensors']:
            if p.exists():
                p.unlink()
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    print('Downloading public 0.86 baseline adapter to Drive fallback...')
    kagglehub.dataset_download('aaitdads/my-0p86-adapter', path='adapter_config.json', output_dir=str(BASELINE_DIR), force_download=True)
    kagglehub.dataset_download('aaitdads/my-0p86-adapter', path='adapter_model.safetensors', output_dir=str(BASELINE_DIR), force_download=True)
    assert adapter_ready(BASELINE_DIR), f'Missing baseline adapter files in {BASELINE_DIR}'
    assert sha256_path(BASELINE_DIR / 'adapter_config.json') == BASE_ADAPTER_CONFIG_SHA256
    assert sha256_path(BASELINE_DIR / 'adapter_model.safetensors') == BASE_ADAPTER_MODEL_SHA256
    return BASELINE_DIR

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if PACK.exists() and sha256_path(PACK) != PACK_SHA256:
    print('Existing Drive pack SHA mismatch; deleting stale pack and downloading the verified one.')
    PACK.unlink()
if not PACK.exists():
    print('Pack not found in Drive; trying GitHub URL...')
    try:
        urllib.request.urlretrieve(PACK_URL, PACK)
    except Exception as exc:
        raise RuntimeError(f'Pack missing. Upload kg1_v198_colab_pack.zip to {PACK} or push the branch so PACK_URL is valid: {PACK_URL}') from exc
pack_hash = sha256_path(PACK)
print('Pack SHA256:', pack_hash)
assert pack_hash == PACK_SHA256, f'Pack SHA mismatch: {pack_hash}'
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK) as zf:
    zf.extractall(ROOT)
assert (ROOT / 'data/v198/v198_micro_train.strict.jsonl').exists()
assert (ROOT / 'data/v198/v198_micro_val.strict.jsonl').exists()
assert (ROOT / 'scripts/hf_job_train_v90.py').exists()

candidates = [
    V195_OUT / 'final_adapter',
    V195_OUT / 'checkpoint-110',
    V195_OUT / 'checkpoint-75',
    V195_OUT / 'checkpoint-55',
]
INIT_ADAPTER = next((p for p in candidates if adapter_ready(p)), None)
if INIT_ADAPTER is None:
    print('No V195 adapter/checkpoint found; falling back to 0.86 baseline adapter.')
    INIT_ADAPTER = ensure_baseline_adapter()
print('INIT_ADAPTER =', INIT_ADAPTER)
print('Pack extracted to', ROOT)


In [ ]:
%cd /content/kg1_v198
import importlib.util, os, subprocess, sys
os.environ.setdefault('MAX_JOBS', '4')
os.environ.setdefault('PIP_ROOT_USER_ACTION', 'ignore')

def pip_install(args):
    print('+ pip install', ' '.join(args))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def pip_uninstall(package_name):
    print('+ pip uninstall -y', package_name)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', package_name], check=False)

def install_if_missing(module_name, args):
    if importlib.util.find_spec(module_name) is None:
        pip_install(args)
    else:
        print(f'{module_name} already installed')

pip_uninstall('torchao')
pip_install(['--upgrade', 'pip', 'setuptools', 'wheel', 'packaging', 'ninja==1.13.0'])
pip_install(['transformers==5.7.0', 'accelerate==1.13.0', 'peft==0.19.1', 'datasets==4.8.5', 'safetensors==0.7.0', 'huggingface_hub==1.13.0', 'sentencepiece==0.2.1', 'protobuf==7.34.1'])
install_if_missing('causal_conv1d', ['causal-conv1d==1.6.1', '--no-build-isolation'])
install_if_missing('mamba_ssm', ['mamba-ssm==2.3.1', '--no-build-isolation'])
assert importlib.util.find_spec('torchao') is None, 'torchao still installed; restart runtime and rerun cells from top'
import causal_conv1d, mamba_ssm
from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn
print('mamba_ssm OK:', getattr(mamba_ssm, '__version__', 'unknown'))


In [ ]:
import os, shutil
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True, exist_ok=True)
os.environ['UPLOAD_TO_HF'] = '0'
os.environ['MODEL_NAME'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
os.environ['DATA_FILE'] = '/content/kg1_v198/data/v198/v198_micro_train.strict.jsonl'
os.environ['VAL_FILE'] = '/content/kg1_v198/data/v198/v198_micro_val.strict.jsonl'
os.environ['INIT_ADAPTER_DIR'] = str(INIT_ADAPTER)
os.environ['INIT_ADAPTER_LOAD_MODE'] = 'manual'
os.environ['PEFT_MANUAL_LOAD_METHOD'] = 'direct'
os.environ['OUTPUT_DIR'] = str(OUT)
os.environ['V198_OUT'] = str(OUT)
os.environ['RUN_ID'] = 'v198-micro-distill-v197-gates'
os.environ['MAX_LENGTH'] = '2048'
os.environ['BATCH_SIZE'] = '16'
os.environ['MICRO_BATCH_SIZE'] = '1'
os.environ['GRADIENT_CHECKPOINTING'] = '1'
os.environ['MAX_STEPS'] = '45'
os.environ['SAVE_EVERY_STEPS'] = '15'
os.environ['EVAL_EVERY_STEPS'] = '15'
os.environ['EVAL_MAX_EXAMPLES'] = '240'
os.environ['LEARNING_RATE'] = '1e-5'
os.environ['FINAL_LEARNING_RATE'] = '3e-6'
os.environ['EXPECTED_TRAIN_SHA256'] = '6d2742616300818eb50c54d36019551b24f5b71c607a2b28feda7461a709def0'
os.environ['EXPECTED_VAL_SHA256'] = 'e59c907c6545e5e587097a64762e3e874508e8cd74d85d5c7c79354ebe56e73c'
os.environ['MIN_TRAIN_EXAMPLES'] = '1875'
os.environ['MIN_TOKENIZED_TRAIN_EXAMPLES'] = '1600'
os.environ['MIN_VAL_EXAMPLES'] = '720'
os.environ['MIN_TOKENIZED_VAL_EXAMPLES'] = '700'
os.environ['TRAINABLE_LORA_MODULES'] = 'in_proj,out_proj,q_proj,k_proj,v_proj,o_proj'
os.environ['MAX_TRAINABLE_PARAM_RATIO'] = '0.035'
!python scripts/hf_job_train_v90.py


Convert the trained adapter to Kaggle layout. This does not submit to Kaggle.


In [ ]:
!python scripts/kg1_convert_local_training_adapter_to_kaggle_zip.py \
  --source-adapter-dir "$V198_OUT/final_adapter" \
  --output-dir "$V198_OUT/kaggle_layout" \
  --run-id v198-micro-distill-v197-gates
